# 可信井 log-AI 正演一致分解

本 notebook 以三带实验的 `F30_B050` 为视觉基线，检验其平滑地质形态在正演约束下是否仍成立。每口井在 5 m 模型网格上求解：

```text
min_delta  ||J delta - (d_full - d_background)||²
         + lambda_curv ||D² delta||²
         + lambda_amp  ||delta||²
```

`J` 是现有深度域正演在 `F30_B050 broad` 附近的局部线性化；每个解随后进入完整非线性 depth forward 复核。优化结果线性提升回 0.1 m 井轴，剩余项定义为 `full log-AI - recoverable log-AI`。

这是一项可恢复性对照，不把正演残差自动解释为地质先验，也不逐井选择最漂亮的参数。


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy import signal

repo_root = Path.cwd().resolve()
if not (repo_root / "src").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "src").is_dir():
    raise RuntimeError("Could not locate repository root containing src/.")
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.physics.numpy_backend import forward_depth, velocity_from_ai
from cup.synthetic.core.signal import finite_support_fir, valid_filter_decimate

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 180,
        "axes.grid": True,
        "grid.alpha": 0.20,
        "font.size": 8.5,
    }
)
print(f"Repository: {repo_root}")

In [ ]:
SOURCE_RUN_DIR = (
    repo_root / "experiments" / "well_residual_decomposition" / "results" / "20260810_three_band_scale_space"
)
RUN_ID = "20260810_forward_consistent_projection"
OUTPUT_DIR = repo_root / "experiments" / "well_residual_decomposition" / "results" / RUN_ID
REFERENCE_CANDIDATE = "F30_B050"
MODEL_GRID_INTERVAL_M = 5.0
CONTEXT_TUNING_MULTIPLIER = 2.0
ZOOM_WINDOW_M = 40.0
LAMBDA_GRID = np.logspace(-4.0, 1.0, 11)
FINAL_MULTIPLIERS = {"F050": 0.5, "F100": 1.0, "F200": 2.0}
LAMBDA_AMP = 1e-3
LINEAR_SOLVE_JITTER = 1e-8
FORWARD_OUTPUT_CHUNK_SIZE = 32

for required in (SOURCE_RUN_DIR / "manifest.json", SOURCE_RUN_DIR / "metrics.csv"):
    if not required.exists():
        raise FileNotFoundError(required)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "wells").mkdir(exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
print(f"Source: {SOURCE_RUN_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 1. 固定输入与模型网格

全频井曲线、TVDSS 轴、层位和三带分解来自上一轮 artifact。`F30_B050 broad` 仅作为本轮局部线性化背景；它不是正式 LFM。优化区间覆盖目标层段及两倍调谐尺度上下文，有限支撑投影和 depth forward 与前两轮相同。


In [ ]:
source_manifest = json.loads((SOURCE_RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
source_metrics = pd.read_csv(SOURCE_RUN_DIR / "metrics.csv")
reference_metrics = source_metrics[source_metrics["candidate"].eq(REFERENCE_CANDIDATE)].set_index(
    "well_name", drop=False
)


def resolve_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else repo_root / path


first_manifest_path = resolve_path(source_manifest["source_manifest"])
first_manifest = json.loads(first_manifest_path.read_text(encoding="utf-8"))
forward_inputs_path = resolve_path(first_manifest["inputs"]["forward_inputs"])
forward_inputs = json.loads(forward_inputs_path.read_text(encoding="utf-8"))
wavelet_path = resolve_path(forward_inputs["wavelet"]["path"])
wavelet_frame = pd.read_csv(wavelet_path)
wavelet_time_s = wavelet_frame["time_s"].to_numpy(dtype=np.float64)
wavelet_amplitude = wavelet_frame["amplitude"].to_numpy(dtype=np.float64)
relation = forward_inputs["ai_velocity_relation"]
AI_VP_A = float(relation["a"])
AI_VP_B = float(relation["b"])

wells = {}
prefix = REFERENCE_CANDIDATE.lower()
for well_name in source_manifest["trusted_wells"]:
    artifact_path = resolve_path(source_manifest["well_artifacts"][well_name])
    with np.load(artifact_path, allow_pickle=False) as artifact:
        depth = artifact["tvdss_m"].astype(np.float64)
        full = artifact["well_log_ai"].astype(np.float64)
        valid = artifact["well_valid"].astype(bool)
        horizons = dict(
            zip(
                artifact["horizon_names"].astype(str).tolist(),
                artifact["horizon_tvdss_m"].astype(np.float64).tolist(),
            )
        )
        support = artifact[f"{prefix}_support"].astype(bool)
        background = artifact[f"{prefix}_broad_log_ai"].astype(np.float64)
        reference_detail = artifact[f"{prefix}_geological_detail"].astype(np.float64)
        reference_retained = artifact[f"{prefix}_geological_reconstruction_log_ai"].astype(np.float64)
    dz_m = float(np.median(np.diff(depth)))
    broad_fraction = float(reference_metrics.loc[well_name, "broad_tuning_fraction"])
    tuning_scale_m = float(reference_metrics.loc[well_name, "broad_fwhm_m"]) / broad_fraction
    wells[well_name] = {
        "well_name": well_name,
        "tvdss_m": depth,
        "dz_m": dz_m,
        "full_log_ai": full,
        "valid": valid,
        "horizons": horizons,
        "reference_support": support,
        "background_log_ai": background,
        "reference_detail": reference_detail,
        "reference_retained_log_ai": reference_retained,
        "tuning_scale_m": tuning_scale_m,
        "artifact_path": str(artifact_path),
    }
display(
    pd.DataFrame(
        [
            {
                "well_name": w["well_name"],
                "dz_m": w["dz_m"],
                "tuning_scale_m": w["tuning_scale_m"],
                "target_top_m": min(w["horizons"].values()),
                "target_bottom_m": max(w["horizons"].values()),
            }
            for w in wells.values()
        ]
    )
)

In [ ]:
def finite_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return tuple(slice(int(a), int(b)) for a, b in changes.reshape(-1, 2))


def rms(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.sqrt(np.mean(values**2))) if values.size else np.nan


def exact_forward(model_log_ai, model_depth_m):
    ai = np.exp(model_log_ai)
    vp = velocity_from_ai(ai, a=AI_VP_A, b=AI_VP_B)
    return forward_depth(
        model_log_ai,
        vp,
        model_depth_m,
        wavelet_time_s,
        wavelet_amplitude,
        output_chunk_size=FORWARD_OUTPUT_CHUNK_SIZE,
    )


def prepare_problem(well):
    depth = well["tvdss_m"]
    target_top = float(min(well["horizons"].values()))
    target_bottom = float(max(well["horizons"].values()))
    context_m = CONTEXT_TUNING_MULTIPLIER * well["tuning_scale_m"]
    valid = (
        well["valid"]
        & well["reference_support"]
        & np.isfinite(well["full_log_ai"])
        & np.isfinite(well["background_log_ai"])
    )
    runs = finite_runs(valid)
    if not runs:
        raise ValueError(f"{well['well_name']}: no continuous decomposition support.")
    target_mask = (depth >= target_top) & (depth <= target_bottom)
    run = max(runs, key=lambda item: np.count_nonzero(target_mask[item]))
    start = max(run.start, int(np.searchsorted(depth, target_top - context_m, side="left")))
    stop = min(run.stop, int(np.searchsorted(depth, target_bottom + context_m, side="right")))
    if stop - start < 3:
        raise ValueError(f"{well['well_name']}: optimization interval is too short.")

    factor_float = MODEL_GRID_INTERVAL_M / well["dz_m"]
    factor = int(round(factor_float))
    if factor < 1 or not np.isclose(factor_float, factor, rtol=0.0, atol=1e-6):
        raise ValueError("High-resolution axis is not nested with model grid.")
    taps = finite_support_fir(factor)
    local_full = well["full_log_ai"][start:stop]
    local_background = well["background_log_ai"][start:stop]
    full_model, full_support = valid_filter_decimate(local_full, factor=factor, taps=taps)
    background_model, background_support = valid_filter_decimate(local_background, factor=factor, taps=taps)
    model_depth = depth[start:stop:factor]
    common = full_support & background_support & np.isfinite(full_model) & np.isfinite(background_model)
    common_runs = finite_runs(common)
    if not common_runs:
        raise ValueError(f"{well['well_name']}: no common projected support.")
    model_target = (model_depth >= target_top) & (model_depth <= target_bottom)
    model_run = max(common_runs, key=lambda item: np.count_nonzero(model_target[item]))
    full_model = full_model[model_run]
    background_model = background_model[model_run]
    model_depth = model_depth[model_run]
    model_target = model_target[model_run]
    if np.count_nonzero(model_target) < 5:
        raise ValueError(f"{well['well_name']}: fewer than five target model samples.")
    return {
        "highres_slice": slice(start, stop),
        "model_depth_m": model_depth,
        "full_model_log_ai": full_model,
        "background_model_log_ai": background_model,
        "model_target": model_target,
        "target_top_m": target_top,
        "target_bottom_m": target_bottom,
    }

## 2. 局部线性化、L-curve 与非线性复核

正演权重在背景速度处固定，反射系数对 `log-AI` 的导数保持精确。L-curve 在统一的无量纲目标上扫描相同 λ 网格，并以归一化对数曲线到端点弦线的最大距离选取拐点。最终三档结果均由完整非线性正演重新计算指标。


In [ ]:
def build_linearized_problem(problem):
    background = problem["background_model_log_ai"]
    depth = problem["model_depth_m"]
    target_synthetic = exact_forward(problem["full_model_log_ai"], depth)
    background_ai = np.exp(background)
    background_vp = velocity_from_ai(background_ai, a=AI_VP_A, b=AI_VP_B)
    background_synthetic, wavelet_operator = forward_depth(
        background,
        background_vp,
        depth,
        wavelet_time_s,
        wavelet_amplitude,
        output_chunk_size=FORWARD_OUTPUT_CHUNK_SIZE,
        return_operator=True,
    )
    interface_argument = 0.5 * np.diff(background)
    derivative = 0.5 * (1.0 - np.tanh(interface_argument) ** 2)
    reflectivity_jacobian = np.zeros((background.size - 1, background.size), dtype=np.float64)
    indices = np.arange(background.size - 1)
    reflectivity_jacobian[indices, indices] = -derivative
    reflectivity_jacobian[indices, indices + 1] = derivative
    jacobian = wavelet_operator @ reflectivity_jacobian

    target = problem["model_target"]
    data_scale = max(rms(target_synthetic[target]), 1e-6)
    amplitude_scale = max(float(np.std(problem["full_model_log_ai"][target] - background[target])), 1e-3)
    j_data = jacobian[target] / (data_scale * np.sqrt(np.count_nonzero(target)))
    residual_data = (target_synthetic[target] - background_synthetic[target]) / (
        data_scale * np.sqrt(np.count_nonzero(target))
    )
    n = background.size
    d2 = np.zeros((max(n - 2, 0), n), dtype=np.float64)
    if n >= 3:
        rows = np.arange(n - 2)
        d2[rows, rows] = 1.0
        d2[rows, rows + 1] = -2.0
        d2[rows, rows + 2] = 1.0
    curvature_operator = d2 / (amplitude_scale * np.sqrt(max(n - 2, 1)))
    amplitude_operator = np.eye(n) / (amplitude_scale * np.sqrt(n))
    return {
        "target_synthetic": target_synthetic,
        "background_synthetic": background_synthetic,
        "jacobian_data": j_data,
        "residual_data": residual_data,
        "curvature_operator": curvature_operator,
        "amplitude_operator": amplitude_operator,
        "data_scale": data_scale,
        "amplitude_scale": amplitude_scale,
    }


def solve_delta(linearized, lambda_curv):
    j = linearized["jacobian_data"]
    c = linearized["curvature_operator"]
    a = linearized["amplitude_operator"]
    normal = (
        j.T @ j + float(lambda_curv) * (c.T @ c) + LAMBDA_AMP * (a.T @ a) + LINEAR_SOLVE_JITTER * np.eye(j.shape[1])
    )
    rhs = j.T @ linearized["residual_data"]
    return np.linalg.solve(normal, rhs)


def evaluate_solution(problem, linearized, delta, lambda_curv):
    model = problem["background_model_log_ai"] + delta
    synthetic = exact_forward(model, problem["model_depth_m"])
    target = problem["model_target"]
    residual_synthetic = linearized["target_synthetic"] - synthetic
    curvature = linearized["curvature_operator"] @ delta
    return {
        "lambda_curv": float(lambda_curv),
        "delta": delta,
        "recoverable_model_log_ai": model,
        "synthetic_recoverable": synthetic,
        "synthetic_residual": residual_synthetic,
        "data_nrmse": rms(residual_synthetic[target]) / linearized["data_scale"],
        "roughness": rms(curvature),
        "synthetic_corr": float(np.corrcoef(linearized["target_synthetic"][target], synthetic[target])[0, 1]),
        "residual_synthetic_rms_ratio": (
            rms(residual_synthetic[target]) / max(rms(linearized["target_synthetic"][target]), 1e-12)
        ),
    }


def choose_lcurve_corner(rows):
    data = np.log10(np.clip([row["data_nrmse"] for row in rows], 1e-12, None))
    rough = np.log10(np.clip([row["roughness"] for row in rows], 1e-12, None))
    x = (data - data.min()) / max(float(np.ptp(data)), 1e-12)
    y = (rough - rough.min()) / max(float(np.ptp(rough)), 1e-12)
    start = np.array([x[0], y[0]])
    stop = np.array([x[-1], y[-1]])
    direction = stop - start
    length = max(float(np.linalg.norm(direction)), 1e-12)
    distances = np.abs(direction[0] * (start[1] - y) - (start[0] - x) * direction[1]) / length
    distances[0] = -np.inf
    distances[-1] = -np.inf
    return int(np.argmax(distances)), distances


def lift_model_to_highres(well, problem, model_values):
    output = np.full(well["full_log_ai"].shape, np.nan, dtype=np.float64)
    depth = well["tvdss_m"]
    model_depth = problem["model_depth_m"]
    support = well["reference_support"] & well["valid"] & (depth >= model_depth[0]) & (depth <= model_depth[-1])
    output[support] = np.interp(depth[support], model_depth, model_values)
    return output, support

In [ ]:
all_results = {}
lcurve_rows = []
metric_rows = []
artifact_paths = {}
unavailable_wells = {}

for well_name, well in wells.items():
    try:
        problem = prepare_problem(well)
    except ValueError as error:
        if "no common projected support" not in str(error):
            raise
        status = "unavailable_no_common_projected_support"
        unavailable_wells[well_name] = {"status": status, "error": str(error)}
        for setting, multiplier in FINAL_MULTIPLIERS.items():
            metric_rows.append(
                {
                    "well_name": well_name,
                    "setting": setting,
                    "forward_status": status,
                    "lambda_star": np.nan,
                    "lambda_curv": np.nan,
                    "model_samples": 0,
                    "target_model_samples": 0,
                    "data_nrmse": np.nan,
                    "synthetic_corr": np.nan,
                    "residual_synthetic_rms_ratio": np.nan,
                    "residual_highres_rms": np.nan,
                    "reference_detail_rms": rms(well["reference_detail"][well["reference_support"]]),
                    "residual_reference_detail_corr": np.nan,
                    "reconstruction_max_abs": np.nan,
                }
            )
        print(f"warning: {well_name} | {status}")
        continue
    linearized = build_linearized_problem(problem)
    trials = []
    for lambda_curv in LAMBDA_GRID:
        delta = solve_delta(linearized, float(lambda_curv))
        trial = evaluate_solution(problem, linearized, delta, float(lambda_curv))
        trials.append(trial)
    corner_index, corner_distances = choose_lcurve_corner(trials)
    lambda_star = float(trials[corner_index]["lambda_curv"])
    for index, trial in enumerate(trials):
        lcurve_rows.append(
            {
                "well_name": well_name,
                "lambda_curv": trial["lambda_curv"],
                "data_nrmse": trial["data_nrmse"],
                "roughness": trial["roughness"],
                "synthetic_corr": trial["synthetic_corr"],
                "corner_distance": float(corner_distances[index]),
                "selected_corner": bool(index == corner_index),
            }
        )

    settings = {}
    for setting, multiplier in FINAL_MULTIPLIERS.items():
        lambda_curv = multiplier * lambda_star
        delta = solve_delta(linearized, lambda_curv)
        result = evaluate_solution(problem, linearized, delta, lambda_curv)
        recoverable_highres, support = lift_model_to_highres(well, problem, result["recoverable_model_log_ai"])
        residual_highres = np.where(support, well["full_log_ai"] - recoverable_highres, np.nan)
        reconstructed = np.where(support, recoverable_highres + residual_highres, np.nan)
        parity = float(np.max(np.abs(reconstructed[support] - well["full_log_ai"][support])))
        result.update(
            {
                "lambda_star": lambda_star,
                "support": support,
                "recoverable_highres_log_ai": recoverable_highres,
                "residual_highres_log_ai": residual_highres,
                "reconstruction_max_abs": parity,
            }
        )
        target_highres = (
            support & (well["tvdss_m"] >= problem["target_top_m"]) & (well["tvdss_m"] <= problem["target_bottom_m"])
        )
        metric_rows.append(
            {
                "well_name": well_name,
                "setting": setting,
                "forward_status": "ok",
                "lambda_star": lambda_star,
                "lambda_curv": lambda_curv,
                "model_samples": int(problem["model_depth_m"].size),
                "target_model_samples": int(np.count_nonzero(problem["model_target"])),
                "data_nrmse": result["data_nrmse"],
                "synthetic_corr": result["synthetic_corr"],
                "residual_synthetic_rms_ratio": result["residual_synthetic_rms_ratio"],
                "residual_highres_rms": rms(residual_highres[target_highres]),
                "reference_detail_rms": rms(well["reference_detail"][target_highres]),
                "residual_reference_detail_corr": float(
                    np.corrcoef(residual_highres[target_highres], well["reference_detail"][target_highres])[0, 1]
                ),
                "reconstruction_max_abs": parity,
            }
        )
        settings[setting] = result
    all_results[well_name] = {
        "problem": problem,
        "linearized": linearized,
        "settings": settings,
    }

    artifact_path = OUTPUT_DIR / "wells" / f"{well_name}.npz"
    payload = {
        "tvdss_m": well["tvdss_m"],
        "well_log_ai": well["full_log_ai"],
        "reference_detail": well["reference_detail"],
        "model_depth_m": problem["model_depth_m"],
        "full_model_log_ai": problem["full_model_log_ai"],
        "background_model_log_ai": problem["background_model_log_ai"],
        "target_synthetic": linearized["target_synthetic"],
        "model_target": problem["model_target"],
        "lambda_star": np.asarray(lambda_star),
    }
    for setting, result in settings.items():
        p = setting.lower()
        for key in (
            "support",
            "recoverable_highres_log_ai",
            "residual_highres_log_ai",
            "recoverable_model_log_ai",
            "synthetic_recoverable",
            "synthetic_residual",
        ):
            payload[f"{p}_{key}"] = result[key]
        payload[f"{p}_lambda_curv"] = np.asarray(result["lambda_curv"])
    np.savez_compressed(artifact_path, **payload)
    artifact_paths[well_name] = str(artifact_path)
    print(
        f"completed {well_name} | lambda*={lambda_star:.4g} | "
        f"F100 corr={settings['F100']['synthetic_corr']:.4f} | "
        f"residual ratio={settings['F100']['residual_synthetic_rms_ratio']:.4f}"
    )

metrics = pd.DataFrame(metric_rows).sort_values(["well_name", "setting"])
lcurve = pd.DataFrame(lcurve_rows).sort_values(["well_name", "lambda_curv"])
metrics_path = OUTPUT_DIR / "metrics.csv"
lcurve_path = OUTPUT_DIR / "lcurve.csv"
metrics.to_csv(metrics_path, index=False)
lcurve.to_csv(lcurve_path, index=False)
display(metrics)

## 3. 唯一主图

每口井沿用上一轮参考中间带的高活动短窗口。第一列是 `F30_B050 geological detail`，中间三列是正演一致 residual，最后一列叠加完整井曲线与 `F100` recoverable。所有 residual 列在同一口井内使用相同横轴范围。


In [ ]:
def strongest_window(well, values, support):
    depth = well["tvdss_m"]
    lower = float(min(well["horizons"].values()))
    upper = float(max(well["horizons"].values()))
    eligible = support & np.isfinite(values) & (depth >= lower) & (depth <= upper)
    n = max(5, int(round(ZOOM_WINDOW_M / well["dz_m"])))
    kernel = np.ones(n, dtype=np.float64)
    energy = signal.convolve(np.where(eligible, values**2, 0.0), kernel, mode="same")
    count = signal.convolve(eligible.astype(np.float64), kernel, mode="same")
    energy[count < 0.9 * n] = -np.inf
    center = float(depth[int(np.argmax(energy))]) if np.any(np.isfinite(energy)) else 0.5 * (lower + upper)
    half = 0.5 * min(ZOOM_WINDOW_M, upper - lower)
    if upper - lower >= 2 * half:
        center = float(np.clip(center, lower + half, upper - half))
    return center - half, center + half


def fill_component(axis, values, depth, color):
    axis.plot(values, depth, color=color, linewidth=0.85)
    axis.fill_betweenx(depth, 0.0, values, where=values >= 0.0, color="#c44e52", alpha=0.28)
    axis.fill_betweenx(depth, 0.0, values, where=values < 0.0, color="#4c72b0", alpha=0.28)
    axis.axvline(0.0, color="0.45", linewidth=0.6)


setting_colors = {"F050": "#2ca02c", "F100": "#ff7f0e", "F200": "#d62728"}
fig, axes = plt.subplots(
    len(wells),
    5,
    figsize=(14.5, 2.8 * len(wells)),
    constrained_layout=True,
    squeeze=False,
)
window_manifest = {}
for row, (well_name, well) in enumerate(wells.items()):
    if well_name in all_results:
        settings = all_results[well_name]["settings"]
        window_support = well["reference_support"] & settings["F100"]["support"]
    else:
        settings = None
        window_support = well["reference_support"]
    lower, upper = strongest_window(well, well["reference_detail"], window_support)
    window_manifest[well_name] = [lower, upper]
    depth = well["tvdss_m"]
    view = (depth >= lower) & (depth <= upper)
    reference_values = well["reference_detail"][view & well["reference_support"]]
    if well_name not in all_results:
        residual_limit = float(np.percentile(np.abs(reference_values), 99.0))
        fill_component(axes[row, 0], well["reference_detail"], depth, "#1f77b4")
        axes[row, 0].set_xlim(-residual_limit, residual_limit)
        axes[row, 0].set_ylabel(f"{well_name}\nTVDSS (m)\nforward unavailable")
        for column in (1, 2, 3):
            axes[row, column].text(
                0.5,
                0.5,
                "no common 5 m\nforward support",
                transform=axes[row, column].transAxes,
                ha="center",
                va="center",
                fontsize=7,
            )
        axes[row, 4].plot(well["full_log_ai"], depth, color="0.25", linewidth=0.65, label="full")
        axes[row, 4].set_xlim(*np.percentile(well["full_log_ai"][view & well["valid"]], [1.0, 99.0]))
        axes[row, 4].legend(fontsize=6.2, loc="best")
        for column in range(5):
            axes[row, column].set_ylim(upper, lower)
        continue
    residual_values = [reference_values]
    residual_values.extend(item["residual_highres_log_ai"][view & item["support"]] for item in settings.values())
    residual_limit = float(np.percentile(np.abs(np.concatenate(residual_values)), 99.0))
    fill_component(axes[row, 0], well["reference_detail"], depth, "#1f77b4")
    axes[row, 0].set_xlim(-residual_limit, residual_limit)
    axes[row, 0].set_ylabel(f"{well_name}\nTVDSS (m)\nlambda*={settings['F100']['lambda_star']:.3g}")
    for column, setting in enumerate(("F050", "F100", "F200"), start=1):
        fill_component(
            axes[row, column],
            settings[setting]["residual_highres_log_ai"],
            depth,
            setting_colors[setting],
        )
        axes[row, column].set_xlim(-residual_limit, residual_limit)
    full_limits = np.percentile(well["full_log_ai"][view & well["valid"]], [1.0, 99.0])
    axes[row, 4].plot(well["full_log_ai"], depth, color="0.25", linewidth=0.65, label="full")
    axes[row, 4].plot(
        settings["F100"]["recoverable_highres_log_ai"],
        depth,
        color=setting_colors["F100"],
        linewidth=1.0,
        label="F100 recoverable",
    )
    axes[row, 4].set_xlim(*full_limits)
    axes[row, 4].legend(fontsize=6.2, loc="best")
    for column in range(5):
        axes[row, column].set_ylim(upper, lower)
for column, title in enumerate(
    (
        "F30_B050 geological detail",
        "forward residual F050",
        "forward residual F100",
        "forward residual F200",
        "full vs recoverable",
    )
):
    axes[0, column].set_title(title)
atlas_path = OUTPUT_DIR / "figures" / "forward_consistent_residual_atlas.png"
fig.suptitle("Forward-consistent residual decomposition | common short windows")
fig.savefig(atlas_path, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(atlas_path)))
print(atlas_path)

In [ ]:
review = metrics[["well_name", "setting"]].copy()
for column in (
    "residual_has_geological_texture",
    "same_sign_internal_variation",
    "fixed_wavelength_striping",
    "recoverable_too_smooth",
    "preferred_over_three_band",
    "notes",
):
    review[column] = ""
review_path = OUTPUT_DIR / "human_review.csv"
review.to_csv(review_path, index=False)
manifest = {
    "schema": "well_log_forward_consistent_projection_v1",
    "run_id": RUN_ID,
    "status": "completed_with_warnings" if unavailable_wells else "completed",
    "sample_domain": "depth",
    "sample_unit": "m",
    "depth_basis": "tvdss",
    "source_manifest": str(SOURCE_RUN_DIR / "manifest.json"),
    "reference_candidate": REFERENCE_CANDIDATE,
    "background_semantics": "reference_candidate_broad_not_formal_lfm",
    "model_grid_interval_m": MODEL_GRID_INTERVAL_M,
    "lambda_grid": LAMBDA_GRID.tolist(),
    "lambda_amp": LAMBDA_AMP,
    "final_multipliers": FINAL_MULTIPLIERS,
    "well_artifacts": artifact_paths,
    "unavailable_wells": unavailable_wells,
    "review_windows_tvdss_m": window_manifest,
    "atlas": str(atlas_path),
    "metrics": str(metrics_path),
    "lcurve": str(lcurve_path),
    "human_review": str(review_path),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(
    metrics[
        [
            "well_name",
            "setting",
            "lambda_star",
            "lambda_curv",
            "data_nrmse",
            "synthetic_corr",
            "residual_synthetic_rms_ratio",
            "residual_highres_rms",
            "residual_reference_detail_corr",
        ]
    ]
)
print(f"Manifest: {manifest_path}")
print(f"Human review template: {review_path}")

## 4. 人工判断

先比较第一列与 `F100`：如果正演一致 residual 仍保留非周期、可变厚度且同号内部多峰的结构，这种形态比纯尺度空间结果更可信。如果三档正则化 residual 都退化为密集毛刺，说明上一轮好看的中间带大部分属于地震可恢复成分，亚调谐先验需要从井统计模型而不是直接 residual 曲线中构造。
